# Lambda Search — SSM Loss Weight Experiment

Grid search по весу λ в SSM-лоссе: **λ ∈ {1, 5, 10, 20}**, 30 эпох каждый.

## Перед запуском
1. **Runtime → Change runtime type → T4 GPU**
2. Загрузи `combined_train.pt` в Google Drive (один раз, путь ниже).
3. Запускай ячейки по порядку.

In [ ]:
# ── 1. Проверка GPU ────────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU недоступен — смени runtime на T4")

In [ ]:
# ── 2. Клонируем репозиторий ───────────────────────────────────────────────
import os

REPO_URL  = "https://github.com/valerizabby/sing-learned-structure.git"
REPO_DIR  = "/content/sing-learned-structure"

if os.path.exists(REPO_DIR):
    print("Репо уже есть — делаем git pull")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

print("Done:", REPO_DIR)

In [ ]:
# ── 3. Монтируем Drive (только для данных и сохранения результатов) ────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 5. Установка зависимостей ──────────────────────────────────────────────
!pip install -q sparsemax

In [ ]:
# ── 6. Настройка путей и GPU — ВАЖНО: до импортов SingLS ──────────────────
import sys

os.environ["SING_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
print("SING_DEVICE =", os.environ["SING_DEVICE"])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Перенаправляем все выходные файлы (модели, логи) на Drive
import SingLS.config.config as cfg
cfg.EXP_PATH = RESULTS_ON_DRIVE
print("EXP_PATH =", cfg.EXP_PATH)
print("DEVICE   =", cfg.DEVICE)

In [ ]:
# ── 7. Параметры эксперимента (редактируй здесь) ──────────────────────────
LAMBDA_GRID = [1, 5, 10, 20]
NUM_EPOCHS  = 30
FORCE_RERUN = False   # True = перезапустить даже завершённые прогоны

print(f"λ grid : {LAMBDA_GRID}")
print(f"Epochs : {NUM_EPOCHS}")

In [ ]:
# ── 8. Запуск grid search ──────────────────────────────────────────────────
# Завершённые λ пропускаются автоматически (если FORCE_RERUN=False).
# При падении сессии — просто перезапусти ячейки 6→8.

import logging
from SingLS.config.config import DEVICE, EXP_PATH
from SingLS.config.config import AttentionType, hidden_size, lr, output_size
from SingLS.model.model import MusicGenerator
from SingLS.trainer.train import ModelTrainer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

data = torch.load(
    os.path.join(REPO_DIR, "data/combined/combined_train.pt"),
    weights_only=False,
)
print(f"Data loaded: {len(data)} tracks  |  DEVICE: {DEVICE}")

results = {}

for ssm_lambda in LAMBDA_GRID:
    save_name  = f"sing_lambda_{int(ssm_lambda)}"
    save_dir   = os.path.join(EXP_PATH, f"meta_info/lambda_search/{save_name}")
    final_ckpt = os.path.join(save_dir, "model_final.pt")
    os.makedirs(save_dir, exist_ok=True)

    if not FORCE_RERUN and os.path.exists(final_ckpt):
        print(f"\nSKIP λ={ssm_lambda}: уже обучена → {final_ckpt}")
        results[ssm_lambda] = final_ckpt
        continue

    print(f"\n{'='*60}")
    print(f" RUN λ={ssm_lambda}  ({LAMBDA_GRID.index(ssm_lambda)+1}/{len(LAMBDA_GRID)})")
    print(f"{'='*60}")

    torch.manual_seed(2022)

    model = MusicGenerator(
        hidden_size=hidden_size,
        output_size=output_size,
        attention_type=AttentionType.ORIGINAL,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    trainer = ModelTrainer(
        generator=model,
        optimizer=optimizer,
        data=data,
        hidden_size=hidden_size,
        ssm_lambda=ssm_lambda,
    )

    trainer.train_epochs(
        num_epochs=NUM_EPOCHS,
        full_training=True,
        variable_size_batches=True,
        save_name=save_name,
    )

    torch.save(model, final_ckpt)
    print(f"Saved → {final_ckpt}")
    results[ssm_lambda] = final_ckpt

print("\n" + "="*60)
print("ГОТОВО")
for lam, path in results.items():
    print(f"  λ={lam:<4} → {path}")

In [ ]:
# ── 9. Кривые обучения ─────────────────────────────────────────────────────
import glob
from inference.plot_training import plot_training_curves, print_summary
from IPython.display import Image

csv_files, labels = [], []
for lam in LAMBDA_GRID:
    pattern = os.path.join(EXP_PATH, f"logs/sing_lambda_{int(lam)}_*_metrics.csv")
    found = sorted(glob.glob(pattern))
    if found:
        csv_files.append(found[-1])
        labels.append(f"λ={lam}")
        print_summary(found[-1])

if csv_files:
    out_png = os.path.join(EXP_PATH, "lambda_search_curves.png")
    plot_training_curves(csv_files, labels=labels, out_path=out_png)
    display(Image(out_png))
else:
    print("CSV-логи не найдены — запусти ячейку 8 сначала")

## После обучения

Результаты сохранены на Drive в `sing-results/`:
```
sing-results/
  meta_info/lambda_search/
    sing_lambda_1/model_final.pt
    sing_lambda_5/model_final.pt
    sing_lambda_10/model_final.pt
    sing_lambda_20/model_final.pt
  logs/
    sing_lambda_*_metrics.csv
```

Скачай модели локально и запусти метрики:
```bash
python3 -m inference.quick_eval
```